# Crear el bucket privado de DocIA+ en AWS Academy

Cuaderno para el **Gestor de programaciones del profesorado**. Crea o reutiliza un bucket de propósito general de S3, comprueba su privacidad y configura cifrado SSE-S3.

**Uso en Colab:** Archivo → Subir cuaderno → selecciona este `.ipynb`. Ejecuta las celdas por orden con un entorno **Python 3 / CPU**. No hace falta GPU ni montar Google Drive.

Antes de empezar, inicia tu laboratorio de AWS Academy y abre el apartado que muestra las credenciales temporales (normalmente «AWS Details» / «AWS CLI»). Necesitarás los valores de `aws_access_key_id`, `aws_secret_access_key` y `aws_session_token`, además de la región permitida.

Las credenciales se solicitan mediante **campos ocultos (`getpass`)**, no se escriben en el código ni se guardan en archivos. Se utilizan en la memoria del entorno de ejecución de Colab para autenticar las peticiones a AWS. No imprimas las variables ni compartas el entorno mientras esté conectado. Al terminar, desconecta y elimina el entorno de ejecución.

**Este cuaderno no se ha ejecutado contra tu laboratorio.** Los recursos se crearán cuando tú ejecutes las celdas. S3 puede consumir el presupuesto del laboratorio. No crea usuarios IAM, políticas públicas, servidores, bases de datos ni objetos del corpus.

## 1. Instalar el SDK de AWS
Solo se instala Boto3 en el entorno temporal de Colab. Esta celda no accede a tu cuenta de AWS.

In [ ]:
%pip -q install "boto3>=1.40,<2"

## 2. Funciones de creación y comprobación
Las funciones distinguen `us-east-1`, donde no se envía `LocationConstraint`, del resto de regiones. Si el bucket ya existe y es accesible desde tu cuenta, comprueban su región y reutilizan el bucket sin borrar su contenido. Un error 403 no se interpreta como «no existe».

Un fallo de permisos detiene el paso correspondiente: nunca se transforma en una confirmación de éxito. El laboratorio puede permitir crear buckets y restringir cambios o lecturas de configuración; consulta al docente si sucede.

In [ ]:
import getpass
import re
import secrets
import warnings
import boto3
from botocore.config import Config
from botocore.exceptions import BotoCoreError, ClientError

class ErrorCuaderno(Exception):
    pass

CONFIG_BLOQUEO = {
    "BlockPublicAcls": True,
    "IgnorePublicAcls": True,
    "BlockPublicPolicy": True,
    "RestrictPublicBuckets": True,
}

CONFIG_SDK = Config(connect_timeout=10, read_timeout=30,
                    retries={"max_attempts": 2, "mode": "standard"})

def explicar_error(error):
    # No mostrar trazas, peticiones firmadas ni el contenido completo del error.
    if isinstance(error, ErrorCuaderno):
        return str(error)
    if isinstance(error, ClientError):
        codigo = error.response.get("Error", {}).get("Code", "ErrorAWS")
        mensajes = {
            "AccessDenied": "Permiso denegado. Comprueba los permisos del laboratorio para esta operación.",
            "403": "Acceso denegado. No se modificará un bucket cuya propiedad no pueda verificarse.",
            "ExpiredToken": "Las credenciales han caducado. Renueva las tres credenciales y repite desde el paso 3.",
            "InvalidToken": "El token de sesión no es válido. Copia de nuevo las tres credenciales.",
            "InvalidClientTokenId": "Credenciales no válidas. Comprueba que las tres pertenecen a la misma sesión.",
            "SignatureDoesNotMatch": "No coincide la firma. Revisa las credenciales y la región.",
            "BucketAlreadyExists": "El nombre ya está ocupado. Elige otro en el paso 4.",
            "BucketAlreadyOwnedByYou": "El bucket ya pertenece a tu cuenta. Repite la comprobación desde el paso 5.",
        }
        return mensajes.get(codigo, "AWS ha rechazado la operación. Revisa permisos, región y estado del laboratorio en su consola.")
    return "No se pudo completar la conexión o la operación. Comprueba la red y los pasos anteriores."

def validar_nombre_bucket(nombre):
    if not 3 <= len(nombre) <= 63 or not re.fullmatch(r"[a-z0-9][a-z0-9-]*[a-z0-9]", nombre):
        raise ErrorCuaderno("Usa 3–63 caracteres: minúsculas, números y guiones. Empieza y termina con letra o número. No uses puntos en este piloto.")
    if nombre.startswith(("xn--", "sthree-", "amzn-s3-demo-")) or nombre.endswith(("-s3alias", "--ol-s3", "--x-s3", "--table-s3")):
        raise ErrorCuaderno("Ese nombre utiliza un prefijo o sufijo reservado. Elige otro.")
    return nombre

def parametros_creacion(nombre, region):
    parametros = {"Bucket": validar_nombre_bucket(nombre), "ObjectOwnership": "BucketOwnerEnforced"}
    if region != "us-east-1":
        parametros["CreateBucketConfiguration"] = {"LocationConstraint": region}
    return parametros

def crear_o_reutilizar(s3, nombre, region, cuenta):
    try:
        respuesta = s3.head_bucket(Bucket=nombre, ExpectedBucketOwner=cuenta)
    except ClientError as error:
        codigo = error.response.get("Error", {}).get("Code", "")
        if codigo not in ("404", "NoSuchBucket", "NotFound"):
            raise
        s3.create_bucket(**parametros_creacion(nombre, region))
        s3.get_waiter("bucket_exists").wait(Bucket=nombre, ExpectedBucketOwner=cuenta,
            WaiterConfig={"Delay": 2, "MaxAttempts": 15})
        return "creado"
    region_real = respuesta.get("BucketRegion") or respuesta.get("ResponseMetadata", {}).get("HTTPHeaders", {}).get("x-amz-bucket-region")
    if not region_real:
        region_real = s3.get_bucket_location(Bucket=nombre, ExpectedBucketOwner=cuenta).get("LocationConstraint") or "us-east-1"
        if region_real == "EU":
            region_real = "eu-west-1"
    if region_real != region:
        raise ErrorCuaderno(f"El bucket está en {region_real}. Selecciona esa región y repite desde el paso 4.")
    return "reutilizado"

def configurar_y_verificar(s3, nombre, cuenta):
    comunes = {"Bucket": nombre, "ExpectedBucketOwner": cuenta}
    try:
        bloqueo = s3.get_public_access_block(**comunes)["PublicAccessBlockConfiguration"]
    except ClientError as error:
        if error.response["Error"]["Code"] != "NoSuchPublicAccessBlockConfiguration":
            raise
        bloqueo = {}
    if not all(bloqueo.get(k) is True for k in CONFIG_BLOQUEO):
        s3.put_public_access_block(**comunes, PublicAccessBlockConfiguration=CONFIG_BLOQUEO)
    try:
        reglas = s3.get_bucket_ownership_controls(**comunes)["OwnershipControls"]["Rules"]
    except ClientError as error:
        if error.response["Error"]["Code"] != "OwnershipControlsNotFoundError":
            raise
        reglas = []
    if not any(r.get("ObjectOwnership") == "BucketOwnerEnforced" for r in reglas):
        s3.put_bucket_ownership_controls(**comunes, OwnershipControls={"Rules": [{"ObjectOwnership": "BucketOwnerEnforced"}]})
    try:
        cifrado = s3.get_bucket_encryption(**comunes)["ServerSideEncryptionConfiguration"]["Rules"]
    except ClientError as error:
        if error.response["Error"]["Code"] != "ServerSideEncryptionConfigurationNotFoundError":
            raise
        cifrado = []
    if not any(r.get("ApplyServerSideEncryptionByDefault", {}).get("SSEAlgorithm") == "AES256" for r in cifrado):
        # No cambiar silenciosamente un bucket existente configurado con KMS.
        if cifrado:
            raise ErrorCuaderno("El bucket usa un cifrado distinto de SSE-S3. Utiliza un bucket nuevo para este piloto; no se cambiará su cifrado existente.")
        s3.put_bucket_encryption(**comunes, ServerSideEncryptionConfiguration={"Rules": [
            {"ApplyServerSideEncryptionByDefault": {"SSEAlgorithm": "AES256"}}]})
    bloqueo = s3.get_public_access_block(**comunes)["PublicAccessBlockConfiguration"]
    propiedad = s3.get_bucket_ownership_controls(**comunes)["OwnershipControls"]["Rules"]
    cifrado = s3.get_bucket_encryption(**comunes)["ServerSideEncryptionConfiguration"]["Rules"]
    if not all(bloqueo.get(k) is True for k in CONFIG_BLOQUEO):
        raise ErrorCuaderno("No se ha confirmado el bloqueo completo de acceso público.")
    if not any(r.get("ObjectOwnership") == "BucketOwnerEnforced" for r in propiedad):
        raise ErrorCuaderno("No se ha confirmado la desactivación de ACL.")
    if not any(r.get("ApplyServerSideEncryptionByDefault", {}).get("SSEAlgorithm") == "AES256" for r in cifrado):
        raise ErrorCuaderno("No se ha confirmado el cifrado SSE-S3.")
    return True

print("Funciones preparadas. Todavía no se ha accedido a AWS.")

## 3. Introducir credenciales temporales y validar la sesión
Pega **solo el valor** de cada campo, sin `aws_access_key_id=`, sin comillas y sin el encabezado `[default]`. Los tres campos se ocultan. Se valida la sesión con STS; no se muestran las claves ni el token.

Si renuevas las credenciales, repite esta celda y los pasos posteriores. No introduzcas credenciales de una cuenta personal por error.

In [ ]:
sesion_aws = None
cliente_s3 = None
cuenta_aws = None
bucket_preparado = False
prueba_correcta = False
bucket_creado_o_verificado = False


def solicitar_sesion():
    # Variables locales: no guardar en os.environ, Drive, .env ni salidas.
    with warnings.catch_warnings():
        warnings.simplefilter("error", getpass.GetPassWarning)
        acceso = getpass.getpass("AWS Access Key ID (oculto): ").strip()
        secreto = getpass.getpass("AWS Secret Access Key (oculto): ").strip()
        token = getpass.getpass("AWS Session Token (oculto): ").strip()
    if not acceso or not secreto or not token:
        raise ErrorCuaderno("Debes introducir las tres credenciales temporales.")
    sesion = boto3.Session(aws_access_key_id=acceso, aws_secret_access_key=secreto,
                          aws_session_token=token)
    return sesion

try:
    region_sts = input("Región del laboratorio [us-east-1]: ").strip() or "us-east-1"
    sesion_aws = solicitar_sesion()
    cuenta_aws = sesion_aws.client("sts", region_name=region_sts, config=CONFIG_SDK).get_caller_identity()["Account"]
    print("Sesión temporal validada. Cuenta terminada en", cuenta_aws[-4:])
except getpass.GetPassWarning:
    sesion_aws = None
    print("No se puede garantizar entrada oculta en este entorno. Abre el cuaderno en Google Colab.")
except (ErrorCuaderno, BotoCoreError, ClientError, ValueError) as error:
    sesion_aws = None
    print(explicar_error(error))

## 4. Elegir nombre y región
El nombre sugerido lleva un sufijo aleatorio para evitar colisiones globales. Puedes introducir el nombre de un bucket de este proyecto para reutilizarlo. En ese caso, el cuaderno comprobará y, si es necesario, reforzará sus controles de privacidad; **elige un bucket dedicado al proyecto**.

No se necesitan carpetas vacías: la aplicación crea `programaciones/originales/` y `programaciones/fichas/` al guardar el primer documento.

In [ ]:
bucket_preparado = False
prueba_correcta = False
bucket_creado_o_verificado = False
if sesion_aws is None:
    print("Primero completa correctamente el paso 3.")
else:
    try:
        REGION = input(f"Región S3 [{region_sts}]: ").strip() or region_sts
        regiones = sesion_aws.get_available_regions("s3")
        if REGION not in regiones:
            raise ErrorCuaderno("Región no reconocida por el SDK. Comprueba la región permitida en tu laboratorio.")
        sugerido = globals().get("BUCKET") or ("docia-programaciones-" + secrets.token_hex(5))
        BUCKET = validar_nombre_bucket(input(f"Nombre del bucket [{sugerido}]: ").strip() or sugerido)
        PREFIX = "programaciones"
        cliente_s3 = sesion_aws.client("s3", region_name=REGION, config=CONFIG_SDK)
        print("Configuración seleccionada:", BUCKET, "·", REGION)
    except (ErrorCuaderno, BotoCoreError, ValueError) as error:
        cliente_s3 = None
        print(explicar_error(error))

## 5. Crear el bucket y comprobar su configuración
**Al ejecutar esta celda se crea el recurso en AWS** si todavía no existe. Configuración esperada:

- Las cuatro opciones de bloqueo de acceso público activadas.
- ACL deshabilitadas (`BucketOwnerEnforced`).
- Cifrado predeterminado SSE-S3 (`AES256`).

No se activa alojamiento web, CORS, versionado ni políticas de acceso público. Si hay un fallo a mitad del paso, el bucket puede haberse creado: corrige el problema y repite con el mismo nombre. No se borra el bucket automáticamente.

Permisos de preparación: creación y consulta de bucket, lectura/escritura de bloqueo público, propiedad y cifrado, además de lectura/escritura/listado de objetos para la aplicación. La política orientativa de la aplicación en `docs/iam-policy.example.json` es de **uso**, no incluye todos los permisos de creación/configuración. AWS Academy puede limitar estas operaciones.

In [ ]:
bucket_preparado = False
prueba_correcta = False
bucket_creado_o_verificado = False
if cliente_s3 is None or sesion_aws is None:
    print("Completa los pasos 3 y 4 antes de crear el bucket.")
else:
    try:
        resultado = crear_o_reutilizar(cliente_s3, BUCKET, REGION, cuenta_aws)
        bucket_creado_o_verificado = True
        bucket_preparado = configurar_y_verificar(cliente_s3, BUCKET, cuenta_aws)
        print(f"Bucket {resultado} y configuración verificada: {BUCKET}")
        print("Acceso público bloqueado · ACL deshabilitadas · Cifrado SSE-S3")
    except (ErrorCuaderno, BotoCoreError, ClientError) as error:
        print("No se ha completado la preparación:", explicar_error(error))
        if bucket_creado_o_verificado:
            print("El bucket existe. Revisa sus permisos/configuración antes de continuar; no se ha eliminado.")

## 6. Prueba de escritura, lectura y listado
Esta celda escribe un pequeño TXT con nombre aleatorio bajo `programaciones/originales/_pruebas/`, lo lee, comprueba su contenido y lo elimina. No crea fichas falsas ni toca programaciones existentes. Si falla la limpieza, muestra la clave exacta para retirarla manualmente.

Es una prueba básica del acceso a objetos; no sustituye la primera subida completa desde la aplicación. Si una política distingue permisos de `originales/` y `fichas/`, también habrá que probar la escritura de fichas desde el gestor.

In [ ]:
prueba_correcta = False
if not bucket_preparado:
    print("Primero completa la configuración del paso 5.")
else:
    clave_prueba = f"{PREFIX}/originales/_pruebas/{secrets.token_hex(12)}.txt"
    contenido_prueba = "Prueba de almacenamiento DocIA+.".encode("utf-8")
    try:
        cliente_s3.put_object(Bucket=BUCKET, Key=clave_prueba, Body=contenido_prueba,
            ContentType="text/plain", ServerSideEncryption="AES256", IfNoneMatch="*", ExpectedBucketOwner=cuenta_aws)
        respuesta_prueba = cliente_s3.get_object(Bucket=BUCKET, Key=clave_prueba, ExpectedBucketOwner=cuenta_aws)
        try:
            recuperado = respuesta_prueba["Body"].read()
        finally:
            respuesta_prueba["Body"].close()
        if recuperado != contenido_prueba:
            raise ErrorCuaderno("El contenido recuperado no coincide.")
        cliente_s3.list_objects_v2(Bucket=BUCKET, Prefix=f"{PREFIX}/fichas/", MaxKeys=1, ExpectedBucketOwner=cuenta_aws)
        prueba_correcta = True
        print("Escritura, lectura e inicio del listado de fichas: correctos.")
    except (ErrorCuaderno, BotoCoreError, ClientError) as error:
        print("Prueba no completada:", explicar_error(error))
    finally:
        # Solo esta clave aleatoria creada para la prueba; nunca borra el bucket.
        try:
            cliente_s3.delete_object(Bucket=BUCKET, Key=clave_prueba, ExpectedBucketOwner=cuenta_aws)
            print("Objeto temporal de la prueba retirado.")
        except (BotoCoreError, ClientError):
            prueba_correcta = False
            print("No se confirmó la limpieza. Revisa exclusivamente esta clave:", clave_prueba)

## 7. Valores para la aplicación
Copia estas tres líneas en el `.env` **local** del proyecto. No contienen secretos. Conserva el resto de opciones de `.env.example` y genera tu propia `SECRET_KEY` siguiendo el README.

Las credenciales temporales para ejecutar la aplicación se configuran después en tu ordenador, mediante el perfil de AWS o tu `.env` privado. No se exportan desde este cuaderno. Deben renovarse cuando caduquen.

In [ ]:
if bucket_preparado:
    print(f"S3_BUCKET={BUCKET}")
    print(f"AWS_DEFAULT_REGION={REGION}")
    print(f"S3_PREFIX={PREFIX}")
    if not globals().get("prueba_correcta", False):
        print("Aviso: la prueba de objetos no se ha completado. Revisa el paso 6 antes de utilizar la aplicación.")
else:
    print("No se ha verificado el bucket. Completa el paso 5.")

## 8. Finalizar la sesión
Esta celda retira las referencias principales a la sesión y a los clientes. **No garantiza un borrado seguro de la memoria de Python**. Para terminar, utiliza la opción de Colab **Desconectar y eliminar entorno de ejecución** (el nombre puede variar según el idioma).

No elimina el bucket ni sus documentos. Antes de finalizar el curso/laboratorio, sigue las indicaciones de conservación y exportación de AWS Academy. Si compartes el cuaderno, borra sus salidas: el código nunca contiene credenciales, pero las salidas pueden incluir el nombre del bucket y datos de diagnóstico.

In [ ]:
sesion_aws = None
cliente_s3 = None
cuenta_aws = None
bucket_preparado = False
prueba_correcta = False
bucket_creado_o_verificado = False
for nombre_temporal in ("respuesta_prueba", "recuperado"):
    globals().pop(nombre_temporal, None)
print("Referencias principales retiradas. Desconecta y elimina ahora el entorno de ejecución de Colab.")

## Referencias y solución de problemas

- **ExpiredToken / InvalidToken:** renueva las tres credenciales en Academy y repite desde el paso 3.
- **AccessDenied / 403:** revisa las restricciones del laboratorio. El cuaderno no modifica IAM ni trata de sortearlas.
- **Nombre ocupado:** selecciona otro nombre en el paso 4.
- **Región incorrecta:** utiliza la región del laboratorio o la del bucket existente.
- **Bucket creado pero configuración no verificada:** no se anuncia como preparado. Revisa su configuración y permisos en la consola.
- **No aparecen carpetas:** es normal hasta que la aplicación suba su primera programación; S3 utiliza prefijos.

Documentación oficial: [CreateBucket](https://docs.aws.amazon.com/boto3/latest/reference/services/s3/client/create_bucket.html), [bloqueo de acceso público](https://docs.aws.amazon.com/boto3/latest/reference/services/s3/client/put_public_access_block.html), [cifrado del bucket](https://docs.aws.amazon.com/boto3/latest/reference/services/s3/client/put_bucket_encryption.html).